# 6-3 loss.backward()와 .grad 확인 — 심화

직접 작성한 코드와 저장된 실행 결과를 정리했습니다.


In [3]:
# 검증 가능 정답 코드
# backward용 loss Tensor는 계산 그래프를 유지하고, 로깅용 float는 별도 변수로 분리해 역할을 섞지 않습니다.
import torch

w = torch.tensor(2.0, requires_grad=True)
pred = 3 * w
loss = (pred - 0.0) ** 2

# float는 기록에만 쓰고, backward는 그래프를 가진 loss Tensor에 호출합니다.
loss_value = loss.item()
loss.backward()

# backward 이후 loss type·log type·leaf gradient를 함께 출력해 값 복사와 그래프 연결의 차이를 확인합니다.
print(f"loss_type={type(loss)}")
print(f"log_type={type(loss_value)}")
print(f"log_value={loss_value:.1f}")
print(f"w_grad={w.grad.item():.1f}")

loss_type=<class 'torch.Tensor'>
log_type=<class 'float'>
log_value=36.0
w_grad=36.0


In [4]:
# 검증 가능 정답 코드
# 각 parameter에서 grad None을 먼저 확인한 다음 shape, finite, norm 순으로 감사해 단절과 0 gradient를 구분합니다.
import torch
import torch.nn as nn

model = nn.Linear(2, 1)
with torch.no_grad():
    model.weight.copy_(torch.tensor([[1.0, -1.0]]))
    model.bias.zero_()

x = torch.tensor([[1.0, 2.0], [3.0, 1.0]])
y = torch.tensor([[0.0], [1.0]])
loss = nn.MSELoss()(model(x), y)
loss.backward()

for name, param in model.named_parameters():
    # None을 0으로 바꾸면 그래프 단절을 숨기므로 명시적으로 실패시킵니다.
    if param.grad is None:
        raise RuntimeError(f"{name}: missing gradient")
    shape_ok = param.grad.shape == param.shape
    finite = bool(torch.isfinite(param.grad).all())
    print(f"{name}: shape_ok={shape_ok}, finite={finite}, norm={param.grad.norm().item():.4f}")
# weight와 bias 결과를 이름별 한 줄로 남겨 norm 0인 bias도 연결된 정상 Tensor임을 확인합니다.

weight: shape_ok=True, finite=True, norm=2.2361
bias: shape_ok=True, finite=True, norm=0.0000


In [5]:
# 검증 가능 정답 코드
# 두 실행은 weight와 loss 식을 고정하고 batch B의 입력 스케일만 10배로 바꿔 원인을 분리합니다.
import torch
import torch.nn as nn

model = nn.Linear(1, 1, bias=False)
with torch.no_grad():
    model.weight.fill_(1.0)

def grad_norm(values):
    model.zero_grad()
    x = torch.tensor(values, dtype=torch.float32).reshape(-1, 1)
    y = torch.zeros_like(x)
    loss = nn.MSELoss()(model(x), y)
    loss.backward()
    return model.weight.grad.norm().item()

norm_a = grad_norm([1.0, 1.0])
norm_b = grad_norm([10.0, 10.0])
# gradient norm 비율 100을 계산해 clipping보다 먼저 B의 원시 단위와 전처리를 조사하도록 조치와 연결합니다.
print(f"norm_A={norm_a:.1f}")
print(f"norm_B={norm_b:.1f}")
print(f"ratio={norm_b / norm_a:.1f}")
print("first_action=inspect batch B input scale")

norm_A=2.0
norm_B=200.0
ratio=100.0
first_action=inspect batch B input scale
